# Feature Engineering — Documentation complète

Ce notebook construit l'ensemble des variables utilisées pour la modélisation de la **prédiction du temps jusqu'au prochain éclair nuage-sol (≤ 20 km)**.

### Structure du notebook
1. Imports & configuration
2. Chargement & typage des données brutes
3. Variables temporelles
4. Délais depuis les derniers éclairs
5. Comptages d'éclairs (fenêtres glissantes)
6. Comptage par type d'éclair
7. Taux d'activité
8. Variables spatiales & azimut
9. Intensité électrique (amplitude)
10. Durée d'alerte & indicateurs de burst
11. Dynamique de déplacement de l'orage
12. Silence orageux & direction
13. Centre de masse de l'orage
14. Construction de la cible
15. Split temporel & export

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

AIRPORTS = ['Ajaccio', 'Bastia', 'Bron', 'Nantes', 'Biarritz', 'Pise']
AIRPORT_COORDS = {
    'Bron':    (4.9389,  45.7294),
    'Bastia':  (9.4837,  42.5527),
    'Ajaccio': (8.8029,  41.9236),
    'Nantes':  (-1.6107, 47.1532),
    'Pise':    (10.399,  43.695),
    'Biarritz':(-1.524,  43.4683),
}
print('✅ Imports OK')

✅ Imports OK


## 2. Chargement & typage des données brutes

Le dataset contient **507 071 éclairs** observés sur 6 aéroports entre 2016 et 2022.

### Variables brutes disponibles
| Variable | Type | Description |
|---|---|---|
| `date` | datetime (UTC) | Horodatage de l'éclair |
| `lon`, `lat` | float | Coordonnées géographiques |
| `dist` | float | Distance à l'aéroport (km) |
| `azimuth` | float | Direction depuis l'aéroport (°) |
| `amplitude` | float | Intensité électrique (kA) |
| `maxis` | float | Nombre de maxima du signal |
| `icloud` | bool | `True` = intra-nuage, `False` = nuage-sol |
| `is_last_lightning_cloud_ground` | bool | Dernier CG de l'alerte |

> **Note :** Les éclairs intra-nuage de Pise en 2016 présentent une anomalie détectée (19 404 éclairs). Ils sont conservés dans `df` mais isolables via `mask_pise_2016`.

In [2]:
path = '..\\data\\segment_alerts_all_airports_train.csv'
df = pd.read_csv(path)
print(f'Shape : {df.shape}')

IDS = ['lightning_id','lightning_airport_id','date','lon','lat','airport','airport_alert_id']
VAR = ['dist', 'azimuth']

# Typage
df['date'] = pd.to_datetime(df['date'], utc=True)
df['icloud'] = df['icloud'].astype(bool)
df['is_last_lightning_cloud_ground'] = df['is_last_lightning_cloud_ground'].astype('boolean')

# Tri chronologique par aéroport (obligatoire pour les rolling)
df = df.sort_values(['airport', 'date']).reset_index(drop=True)

# Anomalie Pise 2016
mask_pise_2016 = (df['airport'] == 'Pise') & (df['date'].dt.year == 2016) & (df['icloud'] == True)
print(f'⚠️  Éclairs intra-nuage Pise 2016 : {mask_pise_2016.sum()} (potentiellement biaisés)')

df.head()

Shape : (507071, 13)
⚠️  Éclairs intra-nuage Pise 2016 : 19404 (potentiellement biaisés)


,lightning_id,lightning_airport_id,date,lon,lat,amplitude,maxis,icloud,dist,azimuth,airport,airport_alert_id,is_last_lightning_cloud_ground
0,1,1,2016-01-02 14:53:36+00:00,9.0559,42.0826,-9.90,0.3,False,27.360653,57.852343,Ajaccio,NaN,<NA>
1,2,2,2016-01-02 14:53:36+00:00,9.0236,42.0953,-3.33,0.2,True,26.383167,52.117828,Ajaccio,NaN,<NA>
2,3,3,2016-01-02 21:22:53+00:00,8.8585,42.0456,-18.68,0.4,True,14.313391,24.500543,Ajaccio,NaN,<NA>
3,4,4,2016-01-02 21:22:53+00:00,8.8517,42.0517,-7.51,0.2,False,14.794117,20.854458,Ajaccio,1.0,False
4,5,5,2016-01-02 21:24:46+00:00,8.8728,42.0494,-6.01,0.2,False,15.124224,29.058471,Ajaccio,1.0,False


## 3. Variables temporelles

On extrait l'**année**, le **mois**, l'**heure** et la **saison** à partir du timestamp.
Ces variables permettent de capturer les cycles saisonniers et diurnes de l'activité orageuse
(les orages convectifs sont concentrés en été, en fin d'après-midi).

Elles seront encodées (one-hot pour `season`, numériques pour les autres).

In [3]:
df['year']   = df['date'].dt.year
df['month']  = df['date'].dt.month
df['hour']   = df['date'].dt.hour
df['season'] = df['month'].map({
    12:'Hiver', 1:'Hiver', 2:'Hiver',
    3:'Printemps', 4:'Printemps', 5:'Printemps',
    6:'Été', 7:'Été', 8:'Été',
    9:'Automne', 10:'Automne', 11:'Automne'
})
VAR += ['month', 'hour']
print('✅ Variables temporelles créées')

✅ Variables temporelles créées


## 4. Délais depuis les derniers éclairs

Ces trois variables mesurent **l'intervalle de temps (en secondes)** depuis :
- le dernier éclair **toutes natures confondues**
- le dernier éclair **nuage-sol (CG)**
- le dernier éclair **intra-nuage (IC)**

Elles encodent la **mémoire récente** de l'activité électrique et sont de bons indicateurs de l'intensité de l'orage.

### Imputation des valeurs manquantes
Le premier éclair de chaque aeroport n'a pas de prédécesseur → valeur manquante imputée à **0** (hypothèse : l'activité vient de démarrer).

### Transformation logarithmique
Les distributions sont très asymétriques (longue queue à droite jusqu'à plusieurs heures).
On applique `log(clip(x, 0, 3600) + 1)` pour :
- **stabiliser la variance**
- **censurer** les délais > 1h (au-delà, l'information n'est plus pertinente pour la prédiction immédiate)
- ramener les valeurs dans une plage exploitable par les modèles linéaires

In [4]:
df = df.sort_values(['airport', 'date'])
# ── Marqueurs de dates ────────────────────────────────────────────────────────
df['date_cg20'] = df['date'].where(~df['airport_alert_id'].isna())
df['date_cg']   = df['date'].where(df['icloud'] == False)
df['date_ic']   = df['date'].where(df['icloud'] == True)

# ── Dernier événement STRICT (shift(+1) avant ffill) ─────────────────────────
df['_last_lightning'] = df.groupby('airport')['date'].shift(1)
df['last_lightning_date'] = df.groupby('airport')['_last_lightning'].ffill()

df['_last_cg20'] = df.groupby('airport')['date_cg20'].shift(1)
df['last_cg20_date'] = df.groupby('airport')['_last_cg20'].ffill()

df['_last_cg'] = df.groupby('airport')['date_cg'].shift(1)
df['last_cg_date'] = df.groupby('airport')['_last_cg'].ffill()

df['_last_ic'] = df.groupby('airport')['date_ic'].shift(1)
df['last_ic_date'] = df.groupby('airport')['_last_ic'].ffill()

df.drop(columns=['_last_lightning', '_last_cg20', '_last_cg', '_last_ic'], inplace=True)

# ── Délais (toujours > 0, jamais 0 sur la ligne courante) ────────────────────
df['time_since_last_lightning']    = (df['date'] - df['last_lightning_date']).dt.total_seconds()
df['time_since_last_CG20']         = (df['date'] - df['last_cg20_date']).dt.total_seconds()
df['time_since_last_cloud_ground'] = (df['date'] - df['last_cg_date']).dt.total_seconds()
df['time_since_last_intra_cloud']  = (df['date'] - df['last_ic_date']).dt.total_seconds()



# Prochain CG20 STRICT : shift(-1) dans le groupe avant bfill
df['_cg20_shifted'] = df.groupby('airport')['date_cg20'].shift(-1)
df['next_cg20_date'] = df.groupby('airport')['_cg20_shifted'].bfill()
df.drop(columns='_cg20_shifted', inplace=True)

df['time_to_next_cg20'] = (df['next_cg20_date'] - df['date']).dt.total_seconds()


print('✅ Délais passés + futur strict créés')



✅ Délais passés + futur strict créés


In [5]:
# Imputation : premier éclair de l'alerte → 0
# df['time_since_last_lightning']    = df['time_since_last_lightning'].fillna(0)
# df['time_since_last_cloud_ground'] = df['time_since_last_cloud_ground'].fillna(0)
# df['time_since_last_intra_cloud']  = df['time_since_last_intra_cloud'].fillna(0)
# df['time_since_last_CG20'] = df['time_since_last_CG20'].fillna(0)
# Transformation log après censure à 1h → stabilise la variance
df['time_since_last_lightning2']    = np.log(df['time_since_last_lightning'].clip(0, 3600) + 1)
df['time_since_last_cloud_ground2'] = np.log(df['time_since_last_cloud_ground'].clip(0, 3600) + 1)
df['time_since_last_intra_cloud2']  = np.log(df['time_since_last_intra_cloud'].clip(0, 3600) + 1)
df['time_since_last_CG20_2']  = np.log(df['time_since_last_CG20'].clip(0, 3600) + 1)
VAR += ['time_since_last_lightning2', 'time_since_last_intra_cloud2', 'time_since_last_cloud_ground2',"time_since_last_CG20_2"]
VAR = list(set(VAR))

## 5. Comptages d'éclairs (fenêtres glissantes)

On compte le **nombre d'éclairs survenus dans les X dernières minutes** (fenêtres : 1, 5, 10, 20, 30 min).
Ces features capturent l'**intensité instantanée** de l'orage à différentes échelles temporelles.

Le calcul utilise un `rolling` sur l'index datetime, groupé par aéroport.

### Transformation logarithmique
Les comptages suivent une distribution de Poisson très asymétrique.
`log(count + 1)` stabilise la variance et évite que les pics d'activité ne dominent l'apprentissage.

> **Variables retenues :** `log_count_1min`, `log_count_5min`, `log_count_10min`, `log_count_20min`, `log_count_30min`

In [6]:
df = df.set_index('date')

for window in ['1min', '5min', '10min', '20min', '30min']:
    df[f'count_{window}'] = (
        df.groupby('airport')['lightning_id']
        .rolling(window)
        .count()
        .reset_index(level=0, drop=True)
    )
    df[f'log_count_{window}'] = np.log(df[f'count_{window}'] + 1)
    VAR.append(f'log_count_{window}')

print('✅ Comptages rolling créés')

✅ Comptages rolling créés


## 6. Comptage par type d'éclair

On distingue les éclairs **intra-nuage (IC)** et **nuage-sol (CG)** dans les fenêtres 5, 10 et 20 min.

La proportion CG/IC est un signal fort : une **augmentation de la fraction CG** dans l'activité
est souvent précurseur d'un renforcement de l'orage au sol.

### Transformation logarithmique
Même raison que pour les comptages globaux : `log(x + 1)` pour gérer l'asymétrie.

> **Variables retenues :** `log_ic_count_{5,10,20}min`, `log_cg_count_{5,10,20}min`

In [7]:
df['cg'] = (df['icloud'] == False).astype(int)
df['ic'] = (df['icloud'] == True).astype(int)

for window in ['5min', '10min', '20min']:
    df[f'ic_count_{window}'] = (
        df.groupby('airport')['ic'].rolling(window).sum()
        .reset_index(level=0, drop=True)
    )
    df[f'cg_count_{window}'] = (
        df.groupby('airport')['cg'].rolling(window).sum()
        .reset_index(level=0, drop=True)
    )
    df[f'log_ic_count_{window}'] = np.log(df[f'ic_count_{window}'] + 1)
    df[f'log_cg_count_{window}'] = np.log(df[f'cg_count_{window}'] + 1)
    VAR += [f'log_ic_count_{window}', f'log_cg_count_{window}']

print('✅ Comptages par type créés')

✅ Comptages par type créés


## 7. Taux d'activité

On dérive plusieurs indicateurs de **dynamique d'activité** à partir des comptages :

| Variable | Formule | Interprétation |
|---|---|---|
| `rate_1min` | `count_1min` | Activité immédiate |
| `rate_5min` | `count_5min / 5` | Activité récente normalisée |
| `rate_10min` | `count_10min / 10` | Activité de fond |
| `rate_trend` | `log(count_10min - count_5min + 1)` | Tendance : l'orage monte ou descend ? |
| `activity_decay` | `rate_5min / (rate_10min + ε)` | Ratio : l'activité accélère ou ralentit ? |
| `activity_acceleration` | `rate_1min - rate_5min` | Variation instantanée du rythme |

In [8]:
df['rate_1min']  = df['count_1min']
df['rate_5min']  = df['count_5min'] / 5
df['rate_10min'] = df['count_10min'] / 10

df['rate_trend']            = np.log(df['count_10min'] - df['count_5min'] + 1)
df['activity_decay']        = df['rate_5min'] / (df['rate_10min'] + 1e-6)
df['activity_acceleration'] = df['rate_1min'] - df['rate_5min']

VAR += ['rate_trend', 'activity_decay', 'activity_acceleration']
print('✅ Taux d\'activité créés')

✅ Taux d'activité créés


## 8. Variables spatiales & Azimut

### Distance
On calcule des statistiques rolling sur la **distance éclair-aéroport** :
- `mean_dist` et `min_dist` sur 1, 5, 10 min → proximité de l'orage
- `distance_trend = mean_dist_1min - mean_dist_10min` → l'orage se rapproche (< 0) ou s'éloigne (> 0)

### Dispersion spatiale
- `std_lat_10min`, `std_lon_10min` → étalement géographique de l'orage sur 10 min
- `storm_spread = std_lat + std_lon` → indicateur global de taille du système orageux

### Azimut
- Moyenne et écart-type de l'azimut sur 1 et 10 min
- `azimuth_change = mean_azimuth_1min - mean_azimuth_10min` → rotation du front orageux

### Imputation des valeurs manquantes
Les `std` sont `NaN` quand la fenêtre ne contient qu'un seul éclair → imputés à **0** (pas de dispersion = éclair isolé).

In [9]:
# Distance rolling
for window in ['1min', '5min', '10min']:
    df[f'mean_dist_{window}'] = (
        df.groupby('airport')['dist'].rolling(window).mean()
        .reset_index(level=0, drop=True)
    )
    df[f'min_dist_{window}'] = (
        df.groupby('airport')['dist'].rolling(window).min()
        .reset_index(level=0, drop=True)
    )
    VAR += [f'mean_dist_{window}', f'min_dist_{window}']

df['distance_trend'] = df['mean_dist_1min'] - df['mean_dist_10min']

# Dispersion spatiale — NaN si un seul éclair dans la fenêtre → imputation à 0
df['std_lat_10min'] = (
    df.groupby('airport')['lat'].rolling('10min').std()
    .reset_index(level=0, drop=True)
).fillna(0)

df['std_lon_10min'] = (
    df.groupby('airport')['lon'].rolling('10min').std()
    .reset_index(level=0, drop=True)
).fillna(0)

df['storm_spread'] = df['std_lat_10min'] + df['std_lon_10min']

# Azimut rolling — NaN si un seul éclair → imputation à 0
for window in ['1min', '10min']:
    df[f'mean_azimuth_{window}'] = (
        df.groupby('airport')['azimuth'].rolling(window).mean()
        .reset_index(level=0, drop=True)
    )
    df[f'std_azimuth_{window}'] = (
        df.groupby('airport')['azimuth'].rolling(window).std()
        .reset_index(level=0, drop=True)
    ).fillna(0)
    VAR += [f'mean_azimuth_{window}', f'std_azimuth_{window}']

df['azimuth_change'] = df['mean_azimuth_1min'] - df['mean_azimuth_10min']

VAR += ['distance_trend', 'std_lat_10min', 'std_lon_10min', 'storm_spread', 'azimuth_change']
print('✅ Variables spatiales & azimut créées')

✅ Variables spatiales & azimut créées


## 9. Intensité électrique (amplitude)

L'amplitude (kA) mesure l'intensité du courant de retour de l'éclair.
On calcule des statistiques rolling sur 1 et 10 min :
- `mean_amplitude`, `max_amplitude` → niveau d'énergie moyen et pics
- `amplitude_change = mean_1min - mean_10min` → l'orage gagne ou perd en intensité
- `std_amplitude_10min` → variabilité de l'intensité sur 10 min

### Transformation logarithmique
`std_amplitude_10min` est très asymétrique (quelques orages extrêmement intenses).
On applique `log(std + 1)` pour stabiliser.

### Imputation
`std_amplitude_10min` est `NaN` quand un seul éclair est présent dans la fenêtre → imputation à **0**.

In [10]:
for window in ['1min', '10min']:
    df[f'mean_amplitude_{window}'] = (
        df.groupby('airport')['amplitude'].rolling(window).mean()
        .reset_index(level=0, drop=True)
    )
    df[f'max_amplitude_{window}'] = (
        df.groupby('airport')['amplitude'].rolling(window).max()
        .reset_index(level=0, drop=True)
    )
    VAR += [f'mean_amplitude_{window}', f'max_amplitude_{window}']

df['amplitude_change'] = df['mean_amplitude_1min'] - df['mean_amplitude_10min']

df['std_amplitude_10min'] = (
    df.groupby('airport')['amplitude'].rolling('10min').std()
    .reset_index(level=0, drop=True)
)
# NaN si un seul éclair dans la fenêtre → imputation à 0, puis log
df['log_std_amplitude_10min'] = np.log(df['std_amplitude_10min'].fillna(0) + 1)

VAR += ['amplitude_change', 'log_std_amplitude_10min']
print('✅ Variables amplitude créées')

✅ Variables amplitude créées


## 10. Durée d'alerte & indicateurs de burst

| Variable | Description |
|---|---|
| `alert_duration` | Temps écoulé depuis le début de l'alerte (clippé à 3600 s) |
| `cg_ratio` | Proportion CG dans les 10 dernières minutes |
| `burst_indicator` | 1 si plus de 5 éclairs dans la dernière minute |

### Imputation
`alert_duration` est `NaN` pour les éclairs hors alerte (`airport_alert_id` manquant) → imputé à **ultérieurement**.

La durée est **clippée à 3600 s** pour éviter que les alertes très longues ne créent des valeurs aberrantes.

In [11]:
df['cg_ratio']       = df['cg_count_10min'] / (df['count_10min'] + 1e-6)
df['burst_indicator'] = (df['count_1min'] > 5).astype(int)

df['date'] = pd.to_datetime(df.index, utc=True)
df['alert_start'] = (
    df.groupby(['airport', 'airport_alert_id'])['date']
    .transform('min')
)
df['alert_duration'] = (
    df.index - df['alert_start']
).dt.total_seconds()

# Imputation à 0 + censure à 1h
df['alert_duration'] = df['alert_duration'].clip(0, 3600)

VAR += ['cg_ratio', 'burst_indicator', 'alert_duration']
print('✅ Variables alerte créées')

✅ Variables alerte créées


## 11. Dynamique de déplacement de l'orage

On estime la **vitesse radiale de l'orage** (rapprochement/éloignement de l'aéroport) :

- `delta_t` : intervalle temporel entre deux éclairs consécutifs (s)
- `delta_dist` : variation de distance entre deux éclairs consécutifs (km)
- `storm_velocity = delta_dist / (delta_t + ε)` : vitesse radiale (km/s)

### Imputation & cohérence
Quand deux éclairs consécutifs sont séparés de **plus d'1 heure** (`time_since_last_lightning ≥ 3600 s`),
ils appartiennent à des événements orageux différents. Dans ce cas, `delta_dist` et `storm_velocity`
sont forcés à **0** pour éviter d'introduire un déplacement fictif entre deux orages distincts.

In [12]:
df['delta_t']    = df.groupby('airport')['date'].diff().dt.total_seconds()
df['delta_dist'] = df.groupby('airport')['dist'].diff()
df['storm_velocity'] = df['delta_dist'] / (df['delta_t'] + 1e-6)

# Mise à 0 si éclair isolé (> 1h d'écart) : pas de continuité avec l'éclair précédent
bol = df['time_since_last_lightning'] >= 3600
df.loc[bol, ['delta_dist', 'storm_velocity']] = 0
df[['delta_dist', 'storm_velocity']] = df[['delta_dist', 'storm_velocity']].fillna(0)

VAR += ['delta_dist', 'storm_velocity']
print('✅ Variables dynamique orage créées')

✅ Variables dynamique orage créées


## 12. Silence orageux & changement de direction

### Silence
`silence_30min = 1` si le dernier éclair remonte à plus de 30 minutes → indicateur de **dissipation de l'orage**.
Utile car il capture un régime radicalement différent (fin d'alerte probable).

### Changement de direction
- `azimuth_diff` : différence d'azimut entre deux éclairs consécutifs (°)
- `storm_direction_change = log(|azimuth_diff| + 1)` : transformation log pour compresser les grands changements

### Imputation & cohérence
Comme pour `delta_dist`, on met `azimuth_diff` à **0** si les deux éclairs sont séparés de plus d'1h.

In [13]:
df['silence_30min'] = (df['time_since_last_lightning'] > 1800).astype(int)

df['azimuth_diff'] = df.groupby('airport')['azimuth'].diff()
df['azimuth_diff'] = df['azimuth_diff'].fillna(0)
df.loc[bol, 'azimuth_diff'] = 0   # cohérence : pas de rotation entre orages distincts

df['storm_direction_change'] = np.log(df['azimuth_diff'].abs() + 1)

VAR += ['silence_30min', 'azimuth_diff', 'storm_direction_change']
print('✅ Silence & direction créés')

✅ Silence & direction créés


## 13. Centre de masse de l'orage

On estime le **centre de masse géographique** de l'orage comme la moyenne des coordonnées des éclairs des 10 dernières minutes.

À partir de ce centre, on dérive :
- `storm_center_distance` : distance euclidienne (°) entre le centre de l'orage et l'aéroport
- `storm_center_move` : déplacement de ce centre entre deux éclairs consécutifs
- `storm_center_velocity` : vitesse de déplacement du centre

Ces variables capturent la **trajectoire globale** du système orageux, indépendamment des fluctuations éclair à éclair.

### Imputation & cohérence
`storm_center_move` est `NaN` pour le premier éclair → imputation à **ulterieure**.
Mis à **0** si éclair isolé (> 1h d'écart).

In [14]:
df['storm_lat_center'] = (
    df.groupby('airport')['lat'].rolling('10min').mean()
    .reset_index(level=0, drop=True)
)
df['storm_lon_center'] = (
    df.groupby('airport')['lon'].rolling('10min').mean()
    .reset_index(level=0, drop=True)
)

df['airport_lat'] = df['airport'].map(lambda x: AIRPORT_COORDS[x][1])
df['airport_lon'] = df['airport'].map(lambda x: AIRPORT_COORDS[x][0])

df['storm_center_distance'] = np.sqrt(
    (df['storm_lat_center'] - df['airport_lat'])**2 +
    (df['storm_lon_center'] - df['airport_lon'])**2
)

df['storm_center_move'] = df.groupby('airport')['storm_center_distance'].diff()
df['storm_center_move'] = df['storm_center_move']
df.loc[bol, 'storm_center_move'] = 0

df['storm_center_velocity'] = df['storm_center_move'] / (df['time_since_last_lightning'] + 1)

VAR += ['storm_center_velocity', 'storm_spread', 'storm_center_distance', 'storm_center_move']
print('✅ Centre de masse de l\'orage créé')

✅ Centre de masse de l'orage créé


## 14. Construction de la cible

La variable cible est le **temps (en secondes) jusqu'au prochain éclair nuage-sol ≤ 20 km**.

### Construction
1. Identifier les éclairs CG dans un rayon de 20 km
2. Pour chaque éclair, chercher la date du prochain CG ≤ 20 km (`bfill`)
3. Calculer la différence temporelle
4. Supprimer les lignes sans événement futur (fin de dataset)

### Censure à 3600 s
Les délais > 1h sont clippés : au-delà, l'information n'est plus opérationnellement pertinente
(une alerte n'est pas déclenchée pour un éclair attendu dans plus d'une heure).

### Formes de la cible
| Variable | Description |
|---|---|
| `time_to_next_cg20` | Délai brut en secondes (clippé à 3600) |
| `target_log_time` | `log(time_to_next_cg20 + 1)` — cible pour la **régression** |
| `target_bins` | Classes 0-4 selon : <5min, <10min, <20min, <30min, >30min — cible pour la **classification** |

In [ ]:
df['is_cloud_ground'] = (df['icloud'] == False).astype(int)
df['cg_20km'] = (~df['icloud']) & (df['dist'] <= 20)

# Supprimer les lignes sans futur CG (fin du dataset)
df = df[df['time_to_next_cg20'].notna()]

# Cible régression
df['target_log_time'] = np.log(df['time_to_next_cg20'] + 1)

# Cible classification (5 classes temporelles)
bins   = [0, 300, 600, 1200, 1800, np.inf]
labels = [0, 1, 2, 3, 4]
df['target_bins'] = pd.cut(df['time_to_next_cg20'], bins=bins, labels=labels, include_lowest=True)

VAR   += ['is_cloud_ground', 'cg_20km']
TARGET = ['time_to_next_cg20', 'target_log_time', 'target_bins']

print(f'✅ Cible créée | {len(df):,} lignes conservées')
print(df['target_bins'].value_counts().sort_index())

✅ Cible créée | 507,029 lignes conservées
target_bins
0    387072
1     31643
2     23683
3     10385
4     54246
Name: count, dtype: int64


## 15. Split temporel & export

On utilise un **split temporel strict** (pas de mélange aléatoire) pour éviter toute fuite de données :

| Ensemble | Années | Rôle |
|---|---|---|
| **Train** | 2016–2020 | Apprentissage |
| **Validation** | 2021 | Réglage des hyperparamètres |
| **Test** | 2022 | Évaluation finale (ne pas toucher avant la fin) |

Les variables catégorielles (`season`, `airport`) sont encodées en one-hot.

> **Vérification finale :** `df2[features].isna().sum()` → aucun manquant résiduel.

In [19]:
new_dummies = [
    'season_Automne', 'season_Hiver', 'season_Printemps',
    'airport_Ajaccio', 'airport_Pise', 'airport_Bastia',
    'airport_Biarritz', 'airport_Nantes'
]
df2 = pd.get_dummies(df)
df2[new_dummies] = df2[new_dummies] * 1

features = list(set(VAR)) + new_dummies

# Split temporel strict
train = df2[df2['year'] <= 2020]
val   = df2[df2['year'] == 2021]
test  = df2[df2['year'] >= 2022]

print(f'Train : {train["date"].min()} → {train["date"].max()} | {len(train):,} lignes')
print(f'Val   : {val["date"].min()} → {val["date"].max()} | {len(val):,} lignes')
print(f'Test  : {test["date"].min()} → {test["date"].max()} | {len(test):,} lignes')
print(f'\nNombre de features : {len(features)}')

# Vérification absence de manquants
mis = df2[features].isna().sum()
col_mis = [c for c in mis.index if mis[c] > 0]
if col_mis:
    print('⚠️  Manquants résiduels :', col_mis)
else:
    print('✅ Aucun manquant résiduel')

X_train = train[features] ; y_train = train['target_log_time']
X_val   = val[features]   ; y_val   = val['target_log_time']
X_test  = test[features]  ; y_test  = test['target_log_time']

Train : 2016-01-02 01:10:41+00:00 → 2020-12-31 23:19:36+00:00 | 391,070 lignes
Val   : 2021-01-01 00:06:55+00:00 → 2021-12-27 15:12:35+00:00 | 38,300 lignes
Test  : 2022-01-05 13:32:40+00:00 → 2022-12-16 12:27:12+00:00 | 77,659 lignes

Nombre de features : 64
⚠️  Manquants résiduels : ['time_since_last_CG20_2', 'time_since_last_intra_cloud2', 'time_since_last_cloud_ground2', 'alert_duration', 'storm_center_velocity', 'storm_center_move', 'time_since_last_lightning2']


In [ ]:
import joblib
joblib.dump({
    'df': df,
    'VAR': list(set(VAR)),
    'dumies_vars': new_dummies,
    'TARGET': TARGET,
    'IDS': IDS
}, '../data/meteo_data_clipped.pkl')
print('✅ Données sauvegardées dans ../data/-')

✅ Données sauvegardées dans ../data/meteo_data_clipped.pkl


In [21]:
df.year.value_counts()

year
2018    116033
2016    105074
2022     77659
2017     58983
2020     55680
2019     55300
2021     38300
Name: count, dtype: int64